# 01 - Análise Exploratória (EDA)

Análise dos dados de sinais do EoS Bot para ML.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Config
plt.style.use('dark_background')
pd.set_option('display.max_columns', 50)

DATA_DIR = Path('../data')

## 1. Carregar Dados

In [ ]:
# Primeiro, executar export_data.py se necessário
# !python ../src/export_data.py

df = pd.read_parquet(DATA_DIR / 'signals.parquet')
print(f'Total de sinais: {len(df)}')
df.head()

## 2. Estatísticas Básicas

In [ ]:
print('=== Distribuição WIN/LOSS ===')
print(df['result'].value_counts())
print(f'\nWin Rate: {df["is_win"].mean()*100:.1f}%')

print('\n=== Estatísticas de Profit ===')
print(df['profit_pct'].describe())

In [ ]:
# Distribuição por símbolo
fig, ax = plt.subplots(figsize=(12, 5))
df['symbol'].value_counts().head(15).plot(kind='bar', ax=ax, color='cyan')
ax.set_title('Top 15 Símbolos por Quantidade de Sinais')
ax.set_xlabel('Símbolo')
ax.set_ylabel('Quantidade')
plt.tight_layout()
plt.show()

## 3. Win Rate por Score

In [ ]:
# Win rate por faixa de score
df['score_bin'] = pd.cut(df['score'], bins=[0, 50, 60, 70, 80, 90, 100])
wr_by_score = df.groupby('score_bin')['is_win'].agg(['mean', 'count'])
wr_by_score.columns = ['win_rate', 'count']
wr_by_score['win_rate'] = wr_by_score['win_rate'] * 100
print(wr_by_score)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
wr_by_score['win_rate'].plot(kind='bar', ax=ax, color='lime')
ax.axhline(50, color='red', linestyle='--', label='50% baseline')
ax.set_title('Win Rate por Faixa de Score')
ax.set_xlabel('Score')
ax.set_ylabel('Win Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Análise de Features OFA

In [ ]:
# Features OFA
ofa_features = ['f_smart_money', 'f_price_context', 'f_imbalance', 
                'f_absorption', 'f_trend', 'f_momentum', 'f_volume']

# Correlação com resultado
correlations = df[ofa_features + ['is_win']].corr()['is_win'].drop('is_win').sort_values()
print('=== Correlação Features OFA com WIN ===')
print(correlations)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
correlations.plot(kind='barh', ax=ax, color=['red' if x < 0 else 'lime' for x in correlations])
ax.set_title('Correlação Features OFA com WIN')
ax.set_xlabel('Correlação')
ax.axvline(0, color='white', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Análise Temporal

In [ ]:
# Win rate por hora
wr_by_hour = df.groupby('hour_utc')['is_win'].agg(['mean', 'count'])
wr_by_hour.columns = ['win_rate', 'count']
wr_by_hour['win_rate'] = wr_by_hour['win_rate'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win rate
wr_by_hour['win_rate'].plot(kind='bar', ax=axes[0], color='cyan')
axes[0].axhline(50, color='red', linestyle='--')
axes[0].set_title('Win Rate por Hora (UTC)')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Win Rate (%)')

# Volume de sinais
wr_by_hour['count'].plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Quantidade de Sinais por Hora')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

## 6. Próximos Passos

1. **02_stationarity.ipynb**: Testes ADF e Find Optimal d
2. **03_labeling.ipynb**: Triple Barrier Method
3. **04_feature_eng.ipynb**: X_regime vs X_signal